In [1]:
import pandas as pd
import polars as pl
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import sys

sys.path.append("../../../1_allele_collection/2_analysis")
from plot_utils import *

sys.path.append("../..")
from analyses_plot_utils import *

In [2]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"
meta_output = '../../../1_allele_collection/3_outputs'

In [3]:
orf_distances = pl.read_csv(f'{outputs_dir}/orf_distances.csv', null_values=".", infer_schema_length=10000)
df = pl.read_csv(f'{outputs_dir}/edges_df.csv', null_values=".", infer_schema_length=10000)


In [4]:
# dedup
df = df.with_columns(
    pl.when(pl.col("aa_change").is_null())
    .then(pl.lit("wt"))
    .otherwise(pl.col("aa_change"))
    .alias("aa_change")
)
df = df.with_columns(
    (pl.col("symbol") + "_" + 
     pl.col("ad_symbol") + "_" + 
     pl.col("aa_change")).alias("ppi_group_clean")
)

df = df.sort("db_mut_id", descending=True).unique(subset=["ppi_group_clean"], keep="first")

df

scoring_pla,scoring_pos,large_plate_name,retest_batch,condition,condition_name,assay_id,media_control,ad_orf_id,ad_mut_id,db_orf_id,db_mut_id,db_source_plate,db_source_well,ad_source_plate,ad_source_well,per_well_integrated_intensity_gray,manual_score_growth,total_colony_area,Children_reExpandedColonies_Count,unique_id,manual_score_growth_updated,contamination,outlier_score,outlier_size_score,outlier_flag,size_score_raw,size_score,ad_symbol,symbol,aa_change,gene_variant,spdi,nt_change,chr_num,nuc_loc,ref_allele,…,g2p_Secondary structure (PDBe/SIFTS),g2p_Sequence conflict (UniProt),g2p_Sequence uncertainty (UniProt),g2p_Signal (UniProt),g2p_Site (UniProt),g2p_Substrate genes,g2p_Topological domain (UniProt),g2p_Transit peptide (UniProt),g2p_Transmembrane (UniProt),g2p_Ubiquitination,g2p_Zinc finger (UniProt),plddt_f32,rsa_f32,pLDDT_Category,RSA_Category,OMIM_IDs,inheritance_pattern,inheritance_source,disease_modules,disease_module_count,OMIM_disease_names,splicing_variant,am_pathogenicity,db_group,db_condition_group,ppi_group,edge_group,seq_ad,seq_db,db_seq_final_right,manual_correction_applied,consensus_score,aa_score,consensus_manual_score_growth,wt_consensus_score,log2fc,ppi_group_clean
str,str,str,str,str,str,i64,bool,i64,str,i64,i64,str,str,str,str,f64,f64,i64,i64,i64,f64,bool,bool,bool,bool,f64,f64,str,str,str,str,str,str,f64,f64,str,…,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,f64,str,bool,f64,str,str,str,str,i64,i64,str,bool,f64,f64,f64,f64,f64,str
"""VUSAPWT2B1r05_003""","""E15""","""VUSAPWT2B1r05_001_002_003_004_…","""VUSAPWT2B1""","""r17""","""LWH25""",2,false,6853,null,2713,200818,"""VUSMutY8Dh018""","""G01""","""CEGSint_Y8A2_2""","""F07""",107.579086,0.0,243,1,187597,0.0,false,false,false,false,26141.717849,24.340519,"""MBD4""","""MLH1""","""Leu73Arg""","""MLH1_Leu73Arg""","""NC_000003.12:37000964:T:G""","""218T>G""",3.0,3.7000965e7,"""T""",…,"""Turn""",null,null,null,null,null,null,null,null,null,null,93.79,0.0,""">90""","""<10""","""276300;158320;609310""","""multiple""","""OMIM""","""Rare genetic immune disease; R…",3.0,"""MISMATCH REPAIR CANCER SYNDROM…",false,0.9326,"""2713.0_200818.0_VUSAPWT2B1""","""2713.0_200818.0_LWH25_VUSAPWT2…","""6853.0_2713.0_200818.0_VUSAPWT…","""6853.0_2713.0_VUSAPWT2B1""",1,1,"""Y""",false,21.386004,45.977576,0.0,554.856732,-4.634044,"""MLH1_MBD4_Leu73Arg"""
"""VUSAPWT6B1r05_023""","""O15""","""VUSAPWT6B1r05_019_020_021_022_…","""VUSAPWT6B1""","""r17""","""LWH25""",6,false,70989,null,4062,201430,"""VUSMutY8Dh028""","""C08""","""CEGSint_Y8A6_1""","""D10""",177.979087,0.0,414,5,252613,0.0,false,false,false,false,73683.341967,68.606464,"""FAM209A""","""PLP1""","""Cys193Tyr""","""PLP1_Cys193Tyr""","""NC_000023.11:103788496:G:A""","""578G>A""",23.0,1.03788497e8,"""G""",…,null,null,null,null,null,null,null,null,null,null,null,92.79,2.2222223,""">90""","""<10""",null,null,null,"""Rare genetic eye disease; Rare…",2.0,null,false,0.9996,"""4062.0_201430.0_VUSAPWT6B1""","""4062.0_201430.0_LWH25_VUSAPWT6…","""70989.0_4062.0_201430.0_VUSAPW…","""70989.0_4062.0_VUSAPWT6B1""",1,1,"""Y""",false,63.21432,17.71193,0.0,563.611782,-3.136292,"""PLP1_FAM209A_Cys193Tyr"""
"""VUSAPWT1B1r05_017""","""C23""","""VUSAPWT1B1r05_013_014_015_016_…","""VUSAPWT1B1""","""r11""","""LWH25""",1,false,7233,null,7652,1824,"""VUSMutY8Dh008""","""F10""","""CEGSint_Y8A_4""","""G11""",41.244445,0.0,94,1,40077,0.0,false,false,false,false,3876.977824,3.609849,"""STX5""","""STXBP1""","""Leu365Phe""","""STXBP1_Leu365Phe""","""NC_000009.12:127673243:C:T""","""1093C>T""",9.0,1.27673244e8,"""C""",…,null,null,null,null,null,null,null,null,null,null,null,97.9,0.0,""">90""","""<10""",null,null,null,"""Rare genetic developmental def…",5.0,null,false,0.9892,"""7652.0_1824.0_VUSAPWT1B1""","""7652.0_1824.0_LWH25_VUSAPWT1B1""","""7233.0_7652.0_1824.0_VUSAPWT1B…","""7233.0_7652.0_VUSAPWT1B1""",1,1,"""Y""",false,2.488849,13.868577,0.0,441.405394,-6.986474,"""STXBP1_STX5_Leu365Phe"""
"""VUSAPWT2B1r05_012""",""

In [5]:
# keep only relevant columns for this notebook

# keep only relevant columns for this notebook
df[[
    'db_mut_id', 
    'symbol', 
    'ad_symbol', 
    'aa_change',
    'log2fc',
    'clinvar_clnsig_clean_pp_strict'
]].write_csv('./tmp/edges_df_luke.csv')

In [6]:
# Tables for Leo

# df.select(['symbol', 'db_orf_id', 'aa_change', 'db_mut_id', 'ad_symbol', 'ad_orf_id', 'clinvar_clnsig_clean', 'consensus_score', 'wt_consensus_score', 'log2fc']).write_csv(f"{outputs_dir}/ppi_level_scores_minimal_Leo.csv")

# orf_distances[['symbol', 'db_orf_id', 'aa_change', 'db_mut_id', 'gene_variant' ,'clinvar_clnsig_clean', 'norm_dist_rms']].write_csv(f"{outputs_dir}/orf_distances_minimal_Leo.csv")

In [7]:
orf_distances.filter(
    pl.col("clinvar_clnsig_clean_pp_strict").is_in(["Pathogenic", "Benign", "Likely pathogenic", "Likely benign"])
).with_columns(
    pl.when(
        pl.col("clinvar_clnsig_clean_pp_strict").is_in(["Pathogenic", "Likely pathogenic"])
    )
    .then(pl.lit("Positive"))
    .otherwise(pl.lit("Negative"))
    .alias("referenceSet"),
    pl.col("norm_dist_rms").alias("score"),
    pl.col("gene_variant").alias("hgvsp")
).select(
    pl.col(["hgvsp", "score", "referenceSet", "norm_dist_rms", "gene_variant", "clinvar_clnsig_clean_pp_strict"])
).write_csv("../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv")

orf_distances.with_columns(
    pl.col("norm_dist_rms").alias("score"),
    pl.col("gene_variant").alias("hgvs_pro")
).select(pl.col(["hgvs_pro", "score"])).write_csv("../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv")

In [8]:
%%bash
source /opt/homebrew/Caskroom/miniconda/base/etc/profile.d/conda.sh
conda activate py310

Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_bcv" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_bcv.log" --bandwidth "bcv" --printTransitions  --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_nrd0" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_nrd0.log" --bandwidth "nrd0" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_default" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_default.log" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_nrd" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_nrd.log" --bandwidth "nrd" --printTransitions --kernel "gaussian" --outlierSuppression 0.1
# Rscript ../../threshold_calibration/calcLLR_ppi.R ../3_outputs/sqY2H/calcLLR/scores/Variants_allScores.csv ../3_outputs/sqY2H/calcLLR/scores/Variants_referenceSet.csv --outfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_sj" --logfile "../3_outputs/sqY2H/calcLLR/output/Variants_LLR_sj.log" --printTransitions --bandwidth "SJ" --kernel "gaussian" --outlierSuppression 0.1

Loading required package: parallel
Loading required package: maveLLR

Attaching package: ‘maveLLR’

The following objects are masked _by_ ‘.GlobalEnv’:

    buildLLR.gauss, buildLLR.kernel, drawDensityLLR

Warning message:
NAs introduced by coercion 


2026-02-18_11:55:22 INFO: Reading input 
2026-02-18_11:55:22 INFO: Calculating densties 


Warning message:
In stats::bw.bcv(data) : minimum occurred at one end of the range


2026-02-18_11:55:23 INFO: Drawing LLR plot 
2026-02-18_11:55:24 INFO: Drawing PRC curve 
  |======================================================================| 100%2026-02-18_11:55:24 INFO: Category transitions:  
0.542 : none -> patho.support
0.643 : patho.support -> patho.moderate
0.715 : patho.moderate -> patho.support
2026-02-18_11:55:24 INFO: Bootstrapping confidence intervals at  10000  iterations 
2026-02-18_11:55:24 INFO: Writing output to file  ../3_outputs/sqY2H/calcLLR/output/Variants_LLR_score_bcv.csv 
2026-02-18_11:55:24 INFO: Done! 


In [9]:
LLR_threshold = 0.542

# add ppi_hit column

orf_distances = orf_distances.with_columns(
    pl.when(pl.col("norm_dist_rms") >= LLR_threshold)
    .then(True)
    .otherwise(False)
    .alias("ppi_hit")
)


cols_to_keep = ['db_orf_id',
 'db_mut_id',
 'num_edges',
 'clinvar_clnsig_clean',
 'norm_dist_rms',
 'symbol',
 'gene_variant',
 'aa_change',
 'clinvar_clnsig_clean_pp_strict',
 'ppi_hit'
 ]

orf_distances = orf_distances.select(cols_to_keep)

orf_distances.write_csv(f'{outputs_dir}/orf_distances_final.csv')